# E2 rigid ligand preparation receiving

Registered prospective qualification for #81. Reproduce both admitted authorities
from exact SDFs; inspection JSON only compares observations, never restores MM.
Assign Q/types once each and prepare with active_torsion_bonds=[] and no new H.
Receive actual H merge, Q transfer, types, stereo, axes, bytes and public readers.
No predetermined reduced atom count. Provider-internal calls are observed
separately from direct caps. Stop the two-member selection on failure; retain
returns and not-attempted cases, without retry/substitution. Use pm/fs/coulomb.
No Vina, energies, searches, comparisons, H generation or H5 persistence.

Env BINDINGMT_E2_ROOT/OUTPUT, BINDINGMT_E2_CANDIDATE_MANIFEST/_SHA256,
BINDINGMT_E2_SOURCE_ARCHIVE/_SHA256. Worker injects registered_installed_check.
External resource supervision required; this receives preparation only, not E2.


In [ ]:
import gzip
import hashlib
import inspect
import json
import os
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from functools import wraps
from pathlib import Path, PurePosixPath

root = Path(os.environ['BINDINGMT_E2_ROOT']).resolve()
output = Path(os.environ['BINDINGMT_E2_OUTPUT']).resolve()
assert not output.exists() and not root.is_relative_to(output)
output.mkdir(parents=True, exist_ok=False)
manifest_path = Path(os.environ['BINDINGMT_E2_CANDIDATE_MANIFEST']).resolve()
manifest_sha = os.environ['BINDINGMT_E2_CANDIDATE_MANIFEST_SHA256']
expected_manifest_sha = (
    '898b26fa60561bb99f823d2dd0c27216cc5771ac7dceacdb103760fed2e22284'
)
expected_candidates = [
    dict(
        ligand_id='p59',
        state_id='e2-5x72-p59',
        path='tests/data/vina_torsions/5x72_ligand_p59H.sdf',
        sha256='ef26c05a198a16972efcf8baae644f6b38fd953561f130ee230626088766b08a',
        expected_cip='R',
        source_stereo_atom_index=7,
    ),
    dict(
        ligand_id='p69',
        state_id='e2-5x72-p69',
        path='tests/data/vina_torsions/5x72_ligand_p69H.sdf',
        sha256='637a992134b2038a0ea3cff0c1f8355ea232720c9eae5347100b7186170fcdf1',
        expected_cip='S',
        source_stereo_atom_index=7,
    ),
]
budgets = dict(
    sdf_native=2,
    native_to_rdkit=2,
    rdkit_to_native=2,
    charge_assignment=2,
    type_assignment=2,
    prepare=2,
    hydrogen_inventory=4,
    explicit_cip=4,
    prepared_readiness=2,
    prepared_self_witness=2,
    to_native=2,
    to_pdbqt=2,
    charge_audit=2,
    read_poses=2,
    compose_maps=2,
    torsion_tree=2,
)
counts = dict.fromkeys(budgets, 0)
observed = dict(
    public_convert_alias=0,
    public_get_alias=0,
    snapshot=0,
    public_cip_alias=0,
    public_readiness_alias=0,
    public_correspondence_alias=0,
    internal_merge=0,
    writer_exports=0,
)
forbidden_attempts = []
ledger = [
    dict(
        position=i + 1,
        ligand_id=c['ligand_id'],
        candidate_state_id=c['state_id'],
        status='not_attempted',
        stages={},
    )
    for i, c in enumerate(expected_candidates)
]
current = None
stage = 'custody'


def file_sha(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def save_json(path, value):
    payload = json.dumps(value, indent=2, allow_nan=False) + '\n'
    with path.open('x') as stream:
        stream.write(payload)


def persist_ledger():
    (output / 'ledger.json').write_text(
        json.dumps(ledger, indent=2, allow_nan=False) + '\n'
    )


def event(operation, **fields):
    with (output / 'operations.jsonl').open('a') as stream:
        stream.write(
            json.dumps(
                dict(
                    operation=operation,
                    stage=stage,
                    candidate=None if current is None else current['ligand_id'],
                    at_utc=datetime.now(timezone.utc).isoformat(),
                    direct_counts=counts.copy(),
                    **fields,
                ),
                allow_nan=False,
            )
            + '\n'
        )


persist_ledger()
try:
    raw_manifest = manifest_path.read_bytes()
    (output / 'candidate_manifest.json').write_bytes(raw_manifest)
    assert (
        manifest_sha
        == expected_manifest_sha
        == hashlib.sha256(raw_manifest).hexdigest()
    )
    candidate_manifest = json.loads(raw_manifest)
    assert candidate_manifest['schema'] == 'bindingmt.e2_ligand_source_candidates@1'
    assert candidate_manifest['owner'] == 'uibcdf/bindingmt#81'
    assert len(candidate_manifest['candidates']) == 2
    assert candidate_manifest['source_license']['spdx'] == 'Apache-2.0'
    assert (
        candidate_manifest['source_license']['commit']
        == '3c65c0b3e6c2c1d183f6a175ecb65e3c5ba91645'
    )
    for actual, expected in zip(candidate_manifest['candidates'], expected_candidates):
        assert {key: actual[key] for key in expected} == expected
    stored_inputs = {str(manifest_path): manifest_sha}
    input_copies = {}
    inputs = output / 'inputs'
    inputs.mkdir()
    assert len(candidate_manifest['files']) == 7
    seen = set()
    for row in candidate_manifest['files']:
        relative = PurePosixPath(row['path'])
        assert not relative.is_absolute() and '..' not in relative.parts
        assert str(relative) == row['path'] and row['path'] not in seen
        seen.add(row['path'])
        path = root / relative
        assert path.resolve().is_relative_to(root)
        data = path.read_bytes()
        copy_path = inputs / relative.name
        copy_path.write_bytes(data)
        assert (
            len(data) == row['bytes']
            and hashlib.sha256(data).hexdigest() == row['sha256']
        )
        stored_inputs[str(path)] = row['sha256']
        input_copies[str(copy_path)] = row['sha256']
    for row in expected_candidates:
        assert stored_inputs[str(root / row['path'])] == row['sha256']
    assert os.sched_getaffinity(0) == {0} and os.getpriority(os.PRIO_PROCESS, 0) >= 10
    meminfo = dict(
        line.split(':', 1) for line in Path('/proc/meminfo').read_text().splitlines()
    )
    assert int(meminfo['MemAvailable'].split()[0]) * 1024 >= 8 * 1024**3
    assert not any(name.split('.')[0] == 'vina' for name in sys.modules)
except BaseException as error:
    save_json(
        output / 'receipt.json',
        dict(
            schema='bindingmt.e2_ligand_preparation_receiving@1',
            owner='uibcdf/bindingmt#81',
            outcome='FAIL',
            stage=stage,
            error_type=type(error).__name__,
            reason=str(error),
            direct_counts=counts,
            denominator=2,
            ledger=ledger,
            retry_permitted=False,
        ),
    )
    raise
# Source records are comparisons only, never molecular restoration inputs.
archive = Path(os.environ['BINDINGMT_E2_SOURCE_ARCHIVE']).resolve()
archive_sha = os.environ['BINDINGMT_E2_SOURCE_ARCHIVE_SHA256']
try:
    archive_manifest_path = archive / 'manifest.json'
    assert (
        archive_sha
        == 'a5a60b65715b466bd04dc5e2bdb41a73489fdf9a13254552870628d387221b8d'
    )
    assert file_sha(archive_manifest_path) == archive_sha
    archive_manifest = json.loads(archive_manifest_path.read_text())
    assert archive_manifest['owner'] == 'uibcdf/bindingmt#81'
    assert (
        archive_manifest['molecular_receiving_outcome']
        == archive_manifest['persistence_outcome']
        == 'PASS'
    )
    stored_inputs[str(archive_manifest_path)] = archive_sha
    (output / 'source_archive_manifest.json').write_bytes(
        archive_manifest_path.read_bytes()
    )
    for row in archive_manifest['files']:
        relative = PurePosixPath(row['path'])
        assert not relative.is_absolute() and '..' not in relative.parts
        path = archive / relative
        assert path.resolve().is_relative_to(archive)
        data = path.read_bytes()
        assert (
            len(data) == row['bytes']
            and hashlib.sha256(data).hexdigest() == row['sha256']
        )
        stored_inputs[str(path)] = row['sha256']
        if str(relative).endswith('.gz'):
            decoded = gzip.decompress(data)
            assert (
                len(decoded) == row['decoded_bytes']
                and hashlib.sha256(decoded).hexdigest() == row['decoded_sha256']
            )
except BaseException as error:
    save_json(
        output / 'receipt.json',
        dict(
            schema='bindingmt.e2_ligand_preparation_receiving@1',
            owner='uibcdf/bindingmt#81',
            outcome='FAIL',
            stage='archive-custody',
            reason=str(error),
            direct_counts=counts,
            ledger=ledger,
            retry_permitted=False,
        ),
    )
    raise


def source_record(ligand, filename):
    original = f'attempts/source-01/output/{ligand}/{filename}'
    rows = [
        r
        for r in archive_manifest['files']
        if r.get('original_path', r['path']) == original
    ]
    assert len(rows) == 1, original
    row = rows[0]
    data = (archive / row['path']).read_bytes()
    assert hashlib.sha256(data).hexdigest() == row['sha256']
    return json.loads(gzip.decompress(data) if row['path'].endswith('.gz') else data)

In [ ]:
class RejectVinaImports:
    def find_spec(self, fullname, path=None, target=None):
        if fullname.split('.')[0] == 'vina':
            forbidden_attempts.append('import:' + fullname)
            raise AssertionError('Vina import is forbidden in source admission')
        return None


import_guard = RejectVinaImports()
sys.meta_path.insert(0, import_guard)
try:
    import importlib.metadata as metadata

    import molsysmt as msm
    import numpy as np
    import pyunitwizard as puw

    import bindingmt as dmt
    from bmt_test_support.qualify_chemical_templates import detached_record, snapshot
except BaseException as error:
    save_json(
        output / 'receipt.json',
        dict(
            schema='bindingmt.e2_ligand_preparation_receiving@1',
            owner='uibcdf/bindingmt#81',
            outcome='FAIL',
            stage='imports',
            error_type=type(error).__name__,
            reason=str(error),
            direct_counts=counts,
            denominator=2,
            ledger=ledger,
            forbidden_attempts=forbidden_attempts,
            retry_permitted=False,
        ),
    )
    raise

patches = []
records = []
live_sources = []
identities = {}
installed_check = globals()['registered_installed_check']
caller_policy = detached_record(puw.configure.report())
integrity = dict(before=False, after=False)


def save(path, value):
    save_json(path, detached_record(value))


def patch(owner, name, replacement):
    original = getattr(owner, name)
    patches.append((owner, name, original))
    setattr(owner, name, replacement)


def watch(function, counter):
    @wraps(function)
    def observed_call(*args, **kwargs):
        observed[counter] += 1
        return function(*args, **kwargs)

    return observed_call


def reject(name):
    def forbidden(*args, **kwargs):
        forbidden_attempts.append(name)
        event('forbidden', operation_name=name)
        raise AssertionError('Forbidden source-admission operation: ' + name)

    return forbidden


def direct(label, budget, function, *args, **kwargs):
    global stage
    stage = label
    assert counts[budget] < budgets[budget], budget
    counts[budget] += 1
    current['stages'][label] = 'started'
    persist_ledger()
    event('started', direct_operation=budget)
    value = function(*args, **kwargs)
    current['stages'][label] = 'returned'
    persist_ledger()
    event('returned', direct_operation=budget)
    return value


def source_identity():
    for name, module in (
        ('bindingmt', dmt),
        ('molsysmt', msm),
        ('numpy', np),
        ('pyunitwizard', puw),
    ):
        path = Path(module.__file__).resolve()
        identities[name] = dict(
            version=metadata.version(name), path=str(path), sha256=file_sha(path)
        )
    for name, function in (
        ('convert', msm.convert),
        ('get', msm.get),
        ('chemical_readiness', msm.physchem.get_chemical_readiness),
        ('cip', msm.physchem.get_cip_stereochemistry),
        ('atom_correspondences', msm.chemical_states.get_atom_correspondences),
        ('snapshot', snapshot),
        ('detached_record', detached_record),
    ):
        function = inspect.unwrap(function)
        path = Path(function.__code__.co_filename).resolve()
        identities[name] = dict(
            module=function.__module__, path=str(path), sha256=file_sha(path)
        )
    helper_sha = file_sha(root / 'bmt_test_support/qualify_chemical_templates.py')
    assert (
        identities['snapshot']['sha256']
        == identities['detached_record']['sha256']
        == helper_sha
    )
    save(output / 'initial_module_identities.json', identities)


export_views = []
live_preparations = []


def inspect_native(system):
    observed['snapshot'] += 1
    result = dict(
        form=msm.get_form(system),
        chemistry_structures=snapshot(system),
        n_atoms=int(msm.get(system, n_atoms=True)),
        n_bonds=int(msm.get(system, n_bonds=True)),
        ids=[str(x) for x in msm.get(system, element='atom', atom_id=True)],
        elements=[
            str(x)
            for x in msm.get(
                system,
                element='atom',
                atom_type=True,
                structure_indices=0,
                chemical_state='structure',
            )
        ],
        coordinates_angstrom=puw.get_value(
            msm.get(system, coordinates=True), to_unit='angstrom'
        ),
        coordinate_unit='angstrom',
        has_partial_charge=bool(msm.has_attribute(system, 'partial_charge')),
        has_atom_ff_type=bool(msm.has_attribute(system, 'atom_ff_type')),
    )
    mechanics = system.molecular_mechanics
    q = (
        msm.get(system, element='atom', partial_charge=True)
        if result['has_partial_charge']
        else None
    )
    if puw.is_quantity(q):
        q = puw.get_value(q, to_unit='elementary_charge')
    result.update(
        mechanics=mechanics.to_dict(),
        charges_e=q,
        charge_unit='elementary_charge',
        types=list(msm.get(system, element='atom', atom_ff_type=True))
        if result['has_atom_ff_type']
        else None,
        partial_charge_assignment=mechanics.partial_charge_assignment,
        atom_type_assignment=mechanics.atom_type_assignment,
    )
    return detached_record(result)


def same_chemistry_geometry(left, right):
    a = detached_record(left['chemistry_structures'])
    b = detached_record(right['chemistry_structures'])
    assert a['structure_units'] == b['structure_units']
    ax = a['structures'].pop('coordinates')
    bx = b['structures'].pop('coordinates')
    np.testing.assert_allclose(ax, bx, rtol=0, atol=1e-13)
    assert a == b
    for field in ('n_atoms', 'n_bonds', 'ids', 'elements', 'coordinate_unit'):
        assert left[field] == right[field], field
    np.testing.assert_allclose(
        left['coordinates_angstrom'], right['coordinates_angstrom'], rtol=0, atol=1e-12
    )


def compare_source_observation(actual, expected):
    same_chemistry_geometry(actual, expected)
    for field in ('form', 'has_partial_charge', 'has_atom_ff_type'):
        assert actual[field] == expected[field], field


def preparation_identities():
    source_identity()
    for name, function in (
        ('charge_assignment', msm.build.assign_partial_charges),
        ('type_assignment', msm.build.assign_autodock_atom_types),
        ('hydrogen_inventory', msm.physchem.get_hydrogen_inventory),
        ('hydrogen_projection', msm.build.merge_nonpolar_hydrogens),
        ('prepare_ligand', dmt.prepare_ligand),
        ('known_source_reader', msm.form.string_pdbqt_text.read_poses),
        ('compose_atom_maps', msm.basic.compose_atom_maps),
    ):
        function = inspect.unwrap(function)
        path = Path(function.__code__.co_filename).resolve()
        identities[name] = dict(
            module=function.__module__, path=str(path), sha256=file_sha(path)
        )
    save(output / 'preparation_api_identities.json', identities)


def install_monitoring():
    original_convert = msm.convert

    @wraps(original_convert)
    def convert(*args, **kwargs):
        observed['public_convert_alias'] += 1
        target = kwargs.get('to_form', 'molsysmt.MolSys')
        if isinstance(target, str) and 'h5msm' in target.lower():
            return reject('H5 persistence')()
        writer = target == 'string:pdbqt_text' and kwargs.get('return_atom_map')
        if writer:
            assert observed['writer_exports'] < 2
            observed['writer_exports'] += 1
        result = original_convert(*args, **kwargs)
        if writer:
            text, mapping = result
            model = args[0] if args else kwargs['molecular_system']
            payload = text.removeprefix('pdbqt_text:')
            directory = output / current['ligand_id']
            (directory / 'prepared.pdbqt').write_text(payload)
            save(directory / 'export_map.json', mapping)
            inspection = inspect_native(model)
            save(directory / 'export_view_inspection.json', inspection)
            export_views.append(
                dict(model=model, inspection=inspection, text=payload, atom_map=mapping)
            )
        return result

    patch(msm, 'convert', convert)
    for owner, name, counter in (
        (msm, 'get', 'public_get_alias'),
        (msm.physchem, 'get_cip_stereochemistry', 'public_cip_alias'),
        (msm.physchem, 'get_chemical_readiness', 'public_readiness_alias'),
        (
            msm.chemical_states,
            'get_atom_correspondences',
            'public_correspondence_alias',
        ),
        (msm.build, 'merge_nonpolar_hydrogens', 'internal_merge'),
    ):
        patch(owner, name, watch(getattr(owner, name), counter))
    for owner, name in (
        (dmt, 'dock'),
        (dmt, 'score'),
        (dmt, 'dock_many'),
        (dmt, 'prepare_receptor'),
        (dmt, 'compare_scores'),
        (dmt.VinaBackend, 'dock'),
        (dmt.VinaBackend, 'score'),
        (msm.build, 'add_missing_hydrogens'),
    ):
        patch(owner, name, reject(name))


def retained_cip(report, index):
    position = list(report['atom_indices']).index(index)
    return report['atom_stereochemistry'][position]


def receive_candidate(candidate):
    global stage
    directory = output / candidate['ligand_id']
    directory.mkdir()
    native, report = direct(
        'sdf_native',
        'sdf_native',
        msm.convert,
        str(inputs / Path(candidate['path']).name),
        to_form='molsysmt.MolSys',
        stereo_engine='rdkit',
        discard_properties=True,
        return_report=True,
    )
    save(directory / 'source_conversion_report.json', asdict(report))
    native_before = inspect_native(native)
    save(directory / 'source_native_inspection.json', native_before)
    live_sources.append((native, native_before))
    rdkit_model, report = direct(
        'native_to_rdkit',
        'native_to_rdkit',
        msm.convert,
        native,
        to_form='rdkit.Mol',
        return_report=True,
    )
    save(directory / 'native_to_rdkit_report.json', asdict(report))
    authority, report = direct(
        'rdkit_to_native',
        'rdkit_to_native',
        msm.convert,
        rdkit_model,
        to_form='molsysmt.MolSys',
        return_report=True,
    )
    save(directory / 'rdkit_to_native_report.json', asdict(report))
    authority_before = inspect_native(authority)
    save(directory / 'authority_native_inspection.json', authority_before)
    live_sources.append((authority, authority_before))
    prior_source = source_record(
        candidate['ligand_id'], 'source_native_inspection.json'
    )
    prior_authority = source_record(
        candidate['ligand_id'], 'authority_native_inspection.json'
    )
    prior_admission = source_record(candidate['ligand_id'], 'admission.json')
    compare_source_observation(native_before, prior_source)
    compare_source_observation(authority_before, prior_authority)
    assert prior_admission['current_authority_admission'] == 'ADMITTED'
    assert prior_admission['source_sha256'] == candidate['sha256']
    assert (
        authority_before['n_atoms'] == 39
        and authority_before['elements'].count('H') == 15
    )
    assert authority_before['ids'] == [str(i) for i in range(1, 40)]
    assert inspect_native(native) == native_before
    save(
        directory / 'source_reproduction_gate.json',
        dict(
            status='PASS',
            source_archive_sha256=archive_sha,
            source_sha256=candidate['sha256'],
            source_and_authority_inspections_compared=True,
            coordinate_unit='angstrom',
            coordinate_tolerance=1e-12,
            initial_to_authority_full_graph_equivalence_claimed=False,
            inspection_json_restored=False,
        ),
    )
    inventory_source = direct(
        'source_hydrogen_inventory',
        'hydrogen_inventory',
        msm.physchem.get_hydrogen_inventory,
        authority,
        selection='all',
        structure_indices=0,
        chemical_state='structure',
    )
    inventory_source = detached_record(inventory_source)
    save(directory / 'source_hydrogen_inventory.json', inventory_source)
    cip_source = direct(
        'source_cip',
        'explicit_cip',
        msm.physchem.get_cip_stereochemistry,
        authority,
        structure_indices=0,
        chemical_state='structure',
        engine='rdkit',
        from_coordinates=False,
    )
    cip_source = detached_record(cip_source)
    save(directory / 'source_cip.json', cip_source)
    assert inventory_source['schema'] == 'molsysmt.hydrogen_inventory@1'
    assert (
        inventory_source['status'] == 'available' and inventory_source['issues'] == []
    )
    assert sum(inventory_source['indexed_hydrogen_counts']) == 15
    assert inventory_source['missing_hydrogen_counts'] == [0] * 39
    assert (
        retained_cip(cip_source, 7)
        == candidate['expected_cip']
        == prior_admission['actual_cip']
    )
    # Neutral formal charges belong to the admitted authority. No zeros are written
    # to molecular fields; zero is the explicit total for this new calculation.
    q_result = direct(
        'charge_assignment',
        'charge_assignment',
        msm.build.assign_partial_charges,
        authority,
        method='gasteiger_marsili',
        structure_indices=0,
        chemical_state='structure',
        expected_total_charge=puw.quantity(0, 'elementary_charge'),
        return_report=True,
    )
    save(directory / 'charge_calculation_report.json', q_result['report'])
    charged = q_result['molecular_system']
    charged_before = inspect_native(charged)
    save(directory / 'charged39_inspection.json', charged_before)
    live_sources.append((charged, charged_before))
    assert q_result['report']['schema'] == 'molsysmt.partial_charge_assignment@1'
    assert q_result['report']['method'] == 'gasteiger_marsili'
    assert q_result['report']['expected_total_charge'] == 0
    assert charged_before['partial_charge_assignment']['status'] == 'assigned'
    assert np.isfinite(charged_before['charges_e']).all()
    assert (
        abs(sum(charged_before['charges_e']))
        <= q_result['report']['total_charge_tolerance']
    )
    same_chemistry_geometry(charged_before, authority_before)
    assert inspect_native(authority) == authority_before
    t_result = direct(
        'type_assignment',
        'type_assignment',
        msm.build.assign_autodock_atom_types,
        charged,
        typing_scheme='autodock4',
        method='chemical_environment',
        structure_indices=0,
        chemical_state='structure',
        return_report=True,
    )
    save(directory / 'type_calculation_report.json', t_result['report'])
    typed = t_result['molecular_system']
    typed_before = inspect_native(typed)
    save(directory / 'typed39_inspection.json', typed_before)
    live_sources.append((typed, typed_before))
    assert t_result['report']['schema'] == 'molsysmt.atom_type_assignment@1'
    assert t_result['report']['rule_version'] == 'chemical_environment@1'
    assert t_result['report']['coverage'] == 'complete'
    assert typed_before['atom_type_assignment']['status'] == 'assigned'
    assert len(typed_before['types']) == 39
    np.testing.assert_array_equal(
        typed_before['charges_e'], charged_before['charges_e']
    )
    same_chemistry_geometry(typed_before, authority_before)
    assert inspect_native(charged) == charged_before
    prepared = direct(
        'prepare_ligand',
        'prepare',
        dmt.prepare_ligand,
        typed,
        selection='all',
        state_id=candidate['state_id'],
        active_torsion_bonds=[],
    )
    declaration_before = detached_record(prepared.to_dict())
    coordinates_before = detached_record(
        dict(
            value=puw.get_value(prepared.coordinates, to_unit='angstrom'),
            unit='angstrom',
        )
    )
    save(directory / 'prepared_declaration.json', declaration_before)
    save(directory / 'prepared_coordinates.json', coordinates_before)
    live_preparations.append((prepared, declaration_before, coordinates_before))
    prepared_source_before = inspect_native(prepared.source_molsys)
    save(directory / 'prepared_source_inspection.json', prepared_source_before)
    live_sources.append((prepared.source_molsys, prepared_source_before))
    prepared_native = direct(
        'prepared_to_native', 'to_native', prepared.to_molecular_system
    )
    prepared_before = inspect_native(prepared_native)
    save(directory / 'prepared_native_inspection.json', prepared_before)
    live_sources.append((prepared_native, prepared_before))
    count = prepared.n_atoms
    assert prepared.torsion_dof == 0 and prepared.pdbqt_atom_indices == list(
        range(count)
    )
    assert (
        prepared.metadata['torsion_policy'] == 'rigid_only'
        and prepared.metadata['active_torsion_bonds'] == []
    )
    assert inspect_native(typed) == typed_before
    projection = prepared.metadata['molecular_projection']
    save(directory / 'projection_report.json', projection)
    retained = projection['output_to_source_atom_indices']
    merged = projection['merged_source_atom_pairs']
    assert projection['schema'] == 'molsysmt.atom_projection@1'
    assert projection['n_source_atoms'] == 39 and projection['n_output_atoms'] == count
    assert projection['omitted_source_atom_indices'] == []
    assert retained == [i for i, t in enumerate(typed_before['types']) if t != 'H']
    assert [h for h, _ in merged] == [
        i for i, t in enumerate(typed_before['types']) if t == 'H'
    ]
    assert prepared_before['ids'] == [typed_before['ids'][i] for i in retained]
    assert prepared_before['elements'] == [
        typed_before['elements'][i] for i in retained
    ]
    assert prepared_before['types'] == [typed_before['types'][i] for i in retained]
    np.testing.assert_allclose(
        prepared_before['coordinates_angstrom'],
        np.asarray(typed_before['coordinates_angstrom'])[:, retained],
        rtol=0,
        atol=1e-12,
    )
    np.testing.assert_allclose(
        puw.get_value(prepared.coordinates, to_unit='angstrom'),
        prepared_before['coordinates_angstrom'][0],
        rtol=0,
        atol=1e-12,
    )
    np.testing.assert_allclose(
        prepared.charges, prepared_before['charges_e'], rtol=0, atol=1e-12
    )
    for output_index, source_index in enumerate(retained):
        expected = typed_before['charges_e'][source_index] + sum(
            typed_before['charges_e'][h]
            for h, parent in merged
            if parent == source_index
        )
        assert (
            abs(prepared.charges[output_index] - expected)
            <= projection['charge_conservation_tolerance']
        )
    assert (
        abs(sum(prepared.charges) - sum(typed_before['charges_e']))
        <= projection['charge_conservation_tolerance']
    )
    assert prepared_before['partial_charge_assignment']['status'] == 'derived'
    assert prepared_before['atom_type_assignment']['status'] == 'derived'
    assert (
        prepared_before['partial_charge_assignment']['source_assignment']
        == prepared.metadata['partial_charge_assignment']
    )
    assert (
        prepared_before['atom_type_assignment']['source_assignment']
        == prepared.metadata['atom_type_assignment']
    )
    assert (
        prepared_before['partial_charge_assignment']['method']
        == 'nonpolar_hydrogen_charge_aggregation'
    )
    same_chemistry_geometry(prepared_before, prepared_source_before)
    assert prepared_before['mechanics'] == prepared_source_before['mechanics']
    assert (
        prepared_before['partial_charge_assignment']
        == prepared_source_before['partial_charge_assignment']
    )
    assert (
        prepared_before['atom_type_assignment']
        == prepared_source_before['atom_type_assignment']
    )

    inventory = direct(
        'prepared_hydrogen_inventory',
        'hydrogen_inventory',
        msm.physchem.get_hydrogen_inventory,
        prepared_native,
        structure_indices=0,
        chemical_state='structure',
    )
    inventory = detached_record(inventory)
    save(directory / 'prepared_hydrogen_inventory.json', inventory)
    cip = direct(
        'prepared_cip',
        'explicit_cip',
        msm.physchem.get_cip_stereochemistry,
        prepared_native,
        structure_indices=0,
        chemical_state='structure',
        engine='rdkit',
        from_coordinates=False,
    )
    cip = detached_record(cip)
    save(directory / 'prepared_cip.json', cip)
    readiness = direct(
        'prepared_readiness',
        'prepared_readiness',
        msm.physchem.get_chemical_readiness,
        prepared_native,
        structure_indices=0,
        chemical_state='structure',
    )
    readiness = detached_record(readiness)
    save(directory / 'prepared_readiness.json', readiness)
    raw_witness = direct(
        'prepared_self_witness',
        'prepared_self_witness',
        msm.chemical_states.get_atom_correspondences,
        prepared_native,
        reference_molecular_system=prepared_native,
        structure_indices=0,
        reference_structure_indices=0,
        chemical_state='structure',
        reference_chemical_state='structure',
        candidate_atom_correspondences=[[[i, i] for i in range(count)]],
        max_matches=1,
    )
    witness = detached_record(raw_witness)
    save(directory / 'prepared_self_witness.json', witness)
    audit = direct(
        'charge_audit', 'charge_audit', dmt.audit_preparation_charges, prepared
    )
    save(directory / 'charge_audit.json', audit)
    assert inventory['schema'] == 'molsysmt.hydrogen_inventory@1'
    assert inventory['status'] == 'available' and inventory['issues'] == []
    assert (
        sum(inventory['total_hydrogen_counts'])
        == sum(inventory_source['total_hydrogen_counts'])
        == 15
    )
    assert (
        sum(inventory['indexed_hydrogen_counts'])
        + sum(inventory['missing_hydrogen_counts'])
        == 15
    )
    for output_index, source_index in enumerate(retained):
        assert (
            inventory['total_hydrogen_counts'][output_index]
            == inventory_source['total_hydrogen_counts'][source_index]
        )
    chiral_index = projection['source_to_output_atom_indices'][7]
    assert (
        chiral_index >= 0
        and retained_cip(cip, chiral_index) == candidate['expected_cip']
    )
    assert cip['evidence'] == 'declared_stereochemistry'
    assert readiness['schema'] == 'molsysmt.chemical_readiness@1'
    assert readiness['connectivity']['declared_completeness'] == 'complete'
    assert (
        readiness['connectivity']['invalid_bond_indices']
        == readiness['connectivity']['crossing_bond_indices']
        == []
    )
    for field in (
        'atom_type',
        'formal_charge',
        'atom_is_aromatic',
        'n_unpaired_electrons',
        'n_implicit_hydrogens',
        'n_explicit_hydrogens',
        'allows_implicit_hydrogens',
        'atom_stereochemistry',
        'covalent_multiplicity',
        'coordinates',
    ):
        assert readiness['fields'][field]['status'] == 'present', field
    assert readiness['fields']['n_implicit_hydrogens']['values'] == [0] * count
    assert witness['schema'] == 'molsysmt.atom_correspondences@1'
    assert (
        witness['search_scope'] == 'supplied_atom_correspondences'
        and witness['search_complete'] is True
    )
    assert witness['atom_correspondences'] == [[[i, i] for i in range(count)]]
    assert witness['source_graph_digest'] == witness['reference_graph_digest']
    assert audit['assessment'] == 'consistent'
    raw_witness['software']['molsysmt'] = 'detachment-control'
    raw_witness['atom_correspondences'][0, 0, 0] = -1
    assert inspect_native(prepared_native) == prepared_before
    export_start = len(export_views)
    text = direct('prepared_to_pdbqt', 'to_pdbqt', prepared.to_pdbqt)
    assert len(export_views) == export_start + 1
    exported = export_views[-1]
    mapping = detached_record(exported['atom_map'])
    live_sources.append((exported['model'], exported['inspection']))
    assert text == exported['text']
    assert mapping['schema'] == 'molsysmt.atom_export@1'
    assert mapping['payload_sha256'] == hashlib.sha256(text.encode()).hexdigest()
    assert mapping['n_source_atoms'] == mapping['n_output_atoms'] == count
    assert mapping['source_atom_ids'] == prepared_before['ids']
    assert mapping['output_to_source_atom_indices'] == list(range(count))
    np.testing.assert_allclose(
        mapping['partial_charge_before_rounding'], prepared.charges, rtol=0, atol=1e-12
    )
    assert mapping['output_atom_ff_types'] == prepared.atom_types
    tree = direct(
        'torsion_tree',
        'torsion_tree',
        msm.form.string_pdbqt_text.get_torsion_tree,
        'pdbqt_text:' + text,
    )
    tree = detached_record(tree)
    save(directory / 'torsion_tree.json', tree)
    assert tree['torsdof'] == 0 and tree['fragment_offsets'] == [0, count]
    assert tree['fragment_atom_indices'] == list(range(count))
    assert tree['branch_atom_pairs'] == tree['branch_fragment_pairs'] == []
    assert tree['atom_ids'] == mapping['output_atom_ids']

    composed = direct(
        'compose_maps',
        'compose_maps',
        msm.basic.compose_atom_maps,
        [projection, exported['atom_map']],
    )
    composed = detached_record(composed)
    save(directory / 'composed_atom_map.json', composed)
    assert composed['schema'] == 'molsysmt.atom_map@1'
    assert composed['n_source_atoms'] == 39 and composed['n_output_atoms'] == count
    assert composed['output_to_source_atom_indices'] == retained
    assert (
        composed['source_to_output_atom_indices']
        == projection['source_to_output_atom_indices']
    )
    for h, parent in merged:
        assert (
            typed_before['elements'][h] == 'H'
            and typed_before['elements'][parent] != 'H'
        )
        assert composed['source_to_output_atom_indices'][h] == -1
        assert (
            composed['source_to_representative_atom_indices'][h]
            == composed['source_to_output_atom_indices'][parent]
        )
    received = direct(
        'pdbqt_read',
        'read_poses',
        msm.form.string_pdbqt_text.read_poses,
        'pdbqt_text:' + text,
        exported['model'],
        exported['atom_map'],
        missing_atoms='error',
        return_report=True,
    )
    save(directory / 'pdbqt_read_report.json', received['report'])
    read_inspection = inspect_native(received['molecular_system'])
    save(directory / 'pdbqt_read_inspection.json', read_inspection)
    read_report = detached_record(received['report'])
    assert read_report['schema'] == 'molsysmt.pdbqt_pose_read@1'
    assert read_report['input_payload_sha256'] == mapping['payload_sha256']
    assert read_report['source_coordinates_reused'] is False
    assert read_report['omitted_source_atom_indices'] == []
    assert read_inspection['ids'] == prepared_before['ids']
    assert (
        not read_inspection['has_partial_charge']
        and not read_inspection['has_atom_ff_type']
    )
    np.testing.assert_allclose(
        read_inspection['coordinates_angstrom'],
        prepared_before['coordinates_angstrom'],
        rtol=0,
        atol=0.000500001,
    )
    assert np.asarray(read_report['partial_charge']).shape == (count,)
    assert np.asarray(mapping['written_partial_charge']).shape == (count,)
    np.testing.assert_allclose(
        read_report['partial_charge'],
        mapping['written_partial_charge'],
        rtol=0,
        atol=1e-12,
    )
    assert read_report['atom_ff_type'] == mapping['output_atom_ff_types']
    assert read_report['charge_unit'] == 'elementary_charge'
    for model, before in live_sources:
        assert inspect_native(model) == before
    for model, declaration, coordinates in live_preparations:
        assert detached_record(model.to_dict()) == declaration
        np.testing.assert_array_equal(
            puw.get_value(model.coordinates, to_unit=coordinates['unit']),
            coordinates['value'],
        )
    record = dict(
        ligand_id=candidate['ligand_id'],
        prepared_state_id=prepared.state_id,
        status='ADMITTED',
        source_sha256=candidate['sha256'],
        source_atoms=39,
        prepared_atoms=count,
        indexed_source_hydrogens=15,
        indexed_prepared_hydrogens=sum(
            1 for e in prepared_before['elements'] if e == 'H'
        ),
        represented_total_hydrogens=15,
        merged_hydrogens=len(merged),
        actual_cip=retained_cip(cip, chiral_index),
        source_stereo_atom_index=7,
        prepared_stereo_atom_index=chiral_index,
        source_property_policy='Explicitly discarded native SD properties; raw input retained',
        charge_method='gasteiger_marsili',
        typing_scheme='autodock4',
        typing_rule_version='chemical_environment@1',
        torsion_dof=0,
        pdbqt_sha256=mapping['payload_sha256'],
        projection_schema=projection['schema'],
        composition_schema=composed['schema'],
        prepared_graph_digest=witness['source_graph_digest'],
        source_to_reduced_full_graph_equivalence_claimed=False,
        coordinate_unit='angstrom',
        native_coordinate_tolerance=1e-12,
        pdbqt_coordinate_tolerance=0.000500001,
        charge_unit='elementary_charge',
        new_hydrogens=0,
        new_energies=0,
        new_searches=0,
        inspection_json_is_restoration_input=False,
        whole_e2_received=False,
    )
    save(directory / 'admission.json', record)
    return record

In [ ]:
failure = None
outcome = 'FAIL'
try:
    installed_check('before-e2-preparation')
    integrity['before'] = True
    preparation_identities()
    with puw.context(
        standard_units=[
            'pm',
            'fs',
            'kelvin',
            'mole',
            'dalton',
            'coulomb',
            'kJ/mol',
            'kJ/(mol*pm)',
            'kJ/(mol*pm**2)',
            'radians',
        ]
    ):
        active_policy = detached_record(puw.configure.report())
        save(output / 'active_unit_policy.json', active_policy)
        install_monitoring()
        for position, candidate in enumerate(expected_candidates):
            current = ledger[position]
            current['status'] = 'attempted'
            persist_ledger()
            record = receive_candidate(candidate)
            records.append(record)
            current['status'] = 'admitted'
            current['admission_record'] = candidate['ligand_id'] + '/admission.json'
            persist_ledger()
            assert detached_record(puw.configure.report()) == active_policy
        stage = 'complete_selection_checks'
        assert len(records) == 2 and counts == budgets
        assert observed['internal_merge'] == observed['writer_exports'] == 2
        assert [row['actual_cip'] for row in records] == ['R', 'S']
        assert len({row['prepared_state_id'] for row in records}) == 2
        assert len({row['prepared_graph_digest'] for row in records}) == 2
        for system, before in live_sources:
            assert inspect_native(system) == before
        for model, declaration, coordinates in live_preparations:
            assert detached_record(model.to_dict()) == declaration
            np.testing.assert_array_equal(
                puw.get_value(model.coordinates, to_unit=coordinates['unit']),
                coordinates['value'],
            )
        for path, expected in {**stored_inputs, **input_copies}.items():
            assert file_sha(Path(path)) == expected
        assert not forbidden_attempts
        assert not any(name.split('.')[0] == 'vina' for name in sys.modules)
        for name in (
            'rdkit',
            'rdkit.rdBase',
            'rdkit.Chem.rdchem',
            'rdkit.Chem.rdCIPLabeler',
            'molsysmt.form.file_sdf.to_molsysmt_MolSys',
            'molsysmt.form.molsysmt_MolSys.to_rdkit_Mol',
            'molsysmt.form.rdkit_Mol.to_molsysmt_MolSys',
        ):
            module = sys.modules[name]
            path = Path(module.__file__).resolve()
            identities[name] = dict(path=str(path), sha256=file_sha(path))
        save(output / 'received_module_identities.json', identities)
        save(
            output / 'prepared_ligands.json',
            dict(
                schema='bindingmt.e2_prepared_ligands@1',
                candidates=records,
                distinct_declared_stereochemical_entities=2,
                experimental_complexes=1,
                source_authority_scope='Current rigid prepared ligands from reproduced admitted authorities; no MM restoration, energy or chemical-validity certificate',
                new_preparations=counts['prepare'],
                new_energies=0,
                new_searches=0,
                whole_e2_received=False,
            ),
        )
        outcome = 'PASS'
except BaseException as error:
    failure = error
    if current is not None:
        current['status'] = 'failed'
        current['failed_stage'] = stage
        persist_ledger()
    save(
        output / 'failure.json',
        dict(
            stage=stage,
            error_type=type(error).__name__,
            reason=str(error),
            code=getattr(error, 'code', None),
            direct_counts=counts,
            observed_public_alias_calls=observed,
            complete_denominator=2,
            retry_permitted=False,
        ),
    )
finally:
    for owner, name, original in reversed(patches):
        setattr(owner, name, original)
    if import_guard in sys.meta_path:
        sys.meta_path.remove(import_guard)
    try:
        assert detached_record(puw.configure.report()) == caller_policy
        installed_check('after-e2-preparation')
        assert not any(name.split('.')[0] == 'vina' for name in sys.modules)
        integrity['after'] = True
    except BaseException as error:
        save(
            output / 'integrity_failure.json',
            dict(error_type=type(error).__name__, reason=str(error)),
        )
        if failure is None:
            failure = error
        else:
            failure.add_note('Final integrity check failed: ' + str(error))
    if failure is not None:
        outcome = 'FAIL'
    persist_ledger()
    save(
        output / 'receipt.json',
        dict(
            schema='bindingmt.e2_ligand_preparation_receiving@1',
            owner='uibcdf/bindingmt#81',
            outcome=outcome,
            candidate_manifest_sha256=manifest_sha,
            source_archive_sha256=archive_sha,
            direct_operation_budgets=budgets,
            direct_operation_counts=counts,
            direct_count_scope='Explicit calls only; provider-internal conversions/CIP are not new caller requests',
            observed_public_alias_calls_and_snapshots=observed,
            observed_count_scope='Instrumented public aliases and qualification snapshots; not every internal provider dispatch',
            source_input_sha256=stored_inputs,
            raw_input_copy_sha256=input_copies,
            source_license=candidate_manifest['source_license'],
            module_identities=identities,
            installed_integrity_checks=integrity,
            caller_unit_policy=caller_policy,
            native_unit_policy='pm/fs/coulomb',
            caller_policy_restored=detached_record(puw.configure.report())
            == caller_policy,
            candidate_denominator=2,
            attempted=sum(row['status'] != 'not_attempted' for row in ledger),
            admitted=sum(row['status'] == 'admitted' for row in ledger),
            returned_prepared_cases=len(records),
            ledger=ledger,
            forbidden_attempts=forbidden_attempts,
            new_preparations=counts['prepare'],
            new_charge_assignments=counts['charge_assignment'],
            new_type_assignments=counts['type_assignment'],
            new_hydrogens=0,
            new_energies=0,
            new_searches=0,
            vina_imported=any(name.split('.')[0] == 'vina' for name in sys.modules),
            h5_persistence_calls=0,
            retries=0,
            retry_permitted=False,
            inspection_json_is_restoration_input=False,
            initial_native_to_authority_full_graph_equivalence_claimed=False,
            actual_source_sdf_interpretation='RDKit-supported wedge/3D interpretation; no additional coordinate stereo inference',
            unassessed_science='Valence, protonation, experimental H, conformation, activity, receptor readiness, screening/affinity; all E2 execution remains pending',
            whole_e2_received=False,
        ),
    )
if failure is not None:
    raise failure